# Train XAI YOLO From Scratch

Notebook nay huan luyen YOLO + XAI ngay tu dau tu checkpoint pretrained, khong fine-tune tu baseline da train xong.
Muc tieu la so sanh cong bang giua baseline `model.train(...)` va pipeline `XAI-guided training` voi cung diem khoi tao.


In [ ]:
!pip install ultralytics==8.4.61

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Thanhmay2406/xai-driven-saliency-loss.git"
CLONE_DIR = Path("/kaggle/working/xai-driven-saliency-loss")

if not CLONE_DIR.exists():
    !git clone {REPO_URL} {CLONE_DIR}
else:
    print(f"Repo already exists: {CLONE_DIR}")
    %cd {CLONE_DIR}
    !git pull --ff-only


In [ ]:
import sys
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working")
DEFAULT_REPO_ROOT = KAGGLE_WORKING / "xai-driven-saliency-loss"
REPO_ROOT = DEFAULT_REPO_ROOT if (DEFAULT_REPO_ROOT / "src").exists() else Path.cwd()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)

CFG = {
    "DATASET_YAML": "/kaggle/input/datasets/thanhmay2406/datasettop/merged_yolo_grouped/dataset.yaml",
    "INIT_WEIGHTS": "yolo11n.pt",
    "PROJECT_DIR": "/kaggle/working/yolo_xai_from_scratch",
    "XAI_RUN_NAME": "xai_from_scratch",
    "IMGSZ": 640,
    "BATCH": 16,
    "DEVICE": 0,
    "WORKERS": 4,
    "EPOCHS": 100,
    "PATIENCE": 20,
    "SEED": 42,
    "LR": 1e-4,
    "WEIGHT_DECAY": 5e-4,
    "XAI_METHOD": "activation",
    "LAMBDA_SALIENCY": 0.01,
    "NUM_CLASSES": 6,
    "USE_PROGRESSIVE_LAMBDA": True,
    "PROGRESSIVE_WARMUP_EPOCHS": 40,
    "GT_MASK_MODE": "soft",
    "SOFT_MASK_SIGMA": 0.35,
    "SHRUNK_BOX_RATIO": 0.7,
}

DATASET_YAML = Path(CFG["DATASET_YAML"])
INIT_WEIGHTS = CFG["INIT_WEIGHTS"]
TRAIN_ARGS = {
    "data": str(DATASET_YAML),
    "project": CFG["PROJECT_DIR"],
    "name": CFG["XAI_RUN_NAME"],
    "epochs": CFG["EPOCHS"],
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
    "workers": CFG["WORKERS"],
    "patience": CFG["PATIENCE"],
    "seed": CFG["SEED"],
}
VAL_ARGS = {
    "data": str(DATASET_YAML),
    "imgsz": CFG["IMGSZ"],
    "batch": CFG["BATCH"],
    "device": CFG["DEVICE"],
}
print("Dataset YAML:", DATASET_YAML)
print("Init weights:", INIT_WEIGHTS)
print("Train args:", TRAIN_ARGS)


In [ ]:
if not DATASET_YAML.exists():
    raise FileNotFoundError(f"Khong tim thay dataset yaml: {DATASET_YAML}")

Path(CFG["PROJECT_DIR"]).mkdir(parents=True, exist_ok=True)


In [ ]:
from ultralytics import YOLO

## Load Initial Pretrained Checkpoint

Cell nay load checkpoint pretrained ban dau (`yolo11n.pt`) de kiem tra diem xuat phat truoc khi train XAI tu dau.


In [ ]:
init_model = YOLO(str(INIT_WEIGHTS))
init_model


In [ ]:
val_results = init_model.val(
    **VAL_ARGS,
    split="val",
)
val_results


In [ ]:
test_results = init_model.val(
    **VAL_ARGS,
    split="test",
)
test_results


## XAI Training From Scratch

Pha nay huan luyen YOLO + XAI ngay tu checkpoint pretrained `yolo11n.pt`, khong di qua buoc fine-tune tu baseline da co.
De so sanh cong bang voi baseline, notebook nay giu cung `TRAIN_ARGS` va custom loop chi khac o viec cong them `saliency_loss`.
Vi XAI dang dung custom loop voi `train_loader`, early stopping phai duoc theo doi thu cong sau moi epoch.

Training trong repo hien tai phai dung `activation` de `saliency_loss` thuc su co gradient.
`Grad-CAM`/`Grad-CAM++`/`EigenCAM` chi dung cho visualization hoac evaluation offline.


In [ ]:
import importlib
import inspect
import src.training as training_pkg
import src.training.yolo_xai_trainer as yolo_xai_trainer_module
import src.xai.activation_attention as activation_attention_module

importlib.reload(activation_attention_module)
importlib.reload(yolo_xai_trainer_module)
importlib.reload(training_pkg)

from src.training import UltralyticsYOLOXAITrainer, UltralyticsYOLOXAITrainerConfig
from src.xai.activation_attention import ActivationAttention

xai_cfg = UltralyticsYOLOXAITrainerConfig(
    xai_method=CFG["XAI_METHOD"],
    lambda_saliency=CFG["LAMBDA_SALIENCY"],
    num_classes=CFG["NUM_CLASSES"],
    use_progressive_lambda=CFG["USE_PROGRESSIVE_LAMBDA"],
    progressive_warmup_epochs=CFG["PROGRESSIVE_WARMUP_EPOCHS"],
    gt_mask_mode=CFG["GT_MASK_MODE"],
    soft_mask_sigma=CFG["SOFT_MASK_SIGMA"],
    shrunk_box_ratio=CFG["SHRUNK_BOX_RATIO"],
)
print("Trainer source:", inspect.getfile(UltralyticsYOLOXAITrainer))
xai_cfg

In [ ]:
import copy
import csv
import json
import random
import time

import numpy as np
import torch
from ultralytics.models.yolo.detect import DetectionTrainer

from src.metrics import energy_in_box, pointing_game_accuracy, saliency_iou

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
torch.manual_seed(CFG["SEED"])
torch.cuda.manual_seed_all(CFG["SEED"])

device_str = str(CFG["DEVICE"])
if device_str == "cpu":
    runtime_device = torch.device("cpu")
else:
    runtime_device = torch.device(f"cuda:{device_str}")

# Reload mot model sach cho training de tranh trang thai fused/inference sau cac buoc val truoc do.
train_yolo = YOLO(str(INIT_WEIGHTS))
xai_model = train_yolo.model.to(runtime_device)
xai_model.requires_grad_(True)
xai_model.train()


def build_eval_model(state_dict: dict[str, torch.Tensor]) -> YOLO:
    eval_yolo = YOLO(str(INIT_WEIGHTS))
    eval_yolo.model.load_state_dict(state_dict)
    return eval_yolo


def extract_eval_metrics(results) -> dict[str, object]:
    results_dict = getattr(results, "results_dict", {}) or {}
    names = getattr(results, "names", {}) or {}
    maps = getattr(results, "maps", None)
    per_class_map50_95 = {}
    if maps is not None:
        for idx, value in enumerate(maps):
            class_name = names.get(idx, str(idx))
            per_class_map50_95[class_name] = float(value)

    return {
        "precision": float(results_dict.get("metrics/precision(B)", 0.0)),
        "recall": float(results_dict.get("metrics/recall(B)", 0.0)),
        "map50": float(results_dict.get("metrics/mAP50(B)", 0.0)),
        "map50_95": float(results_dict.get("metrics/mAP50-95(B)", 0.0)),
        "fitness": float(results_dict.get("fitness", 0.0)),
        "per_class_map50_95": per_class_map50_95,
        "save_dir": str(getattr(results, "save_dir", "")),
        "speed": {key: float(value) for key, value in getattr(results, "speed", {}).items()},
    }


def save_json(payload: object, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)


# Dung DetectionTrainer cua Ultralytics de lay train_loader dung batch dict format YOLO.
detector_trainer = DetectionTrainer(
    overrides={
        **TRAIN_ARGS,
        "model": str(INIT_WEIGHTS),
    }
)
detector_trainer.model = xai_model
detector_trainer.device = runtime_device
detector_trainer.set_model_attributes()
xai_model.args = detector_trainer.args
xai_model.names = detector_trainer.data["names"]
xai_model.nc = detector_trainer.data["nc"]
xai_model.criterion = xai_model.init_criterion()
train_loader = detector_trainer.get_dataloader(
    detector_trainer.data["train"],
    batch_size=TRAIN_ARGS["batch"],
    rank=-1,
    mode="train",
)

optimizer = torch.optim.AdamW(
    xai_model.parameters(),
    lr=CFG["LR"],
    weight_decay=CFG["WEIGHT_DECAY"],
)
trainer = UltralyticsYOLOXAITrainer(model=xai_model, optimizer=optimizer, config=xai_cfg)
print("Resolved XAI class:", type(trainer.xai).__name__)
if not isinstance(trainer.xai, ActivationAttention):
    raise RuntimeError(
        "Notebook dang import sai trainer/XAI implementation. Hay Restart Kernel va chay lai tu dau de dam bao `ActivationAttention` duoc load."
    )

best_metric = float("-inf")
best_state = None
best_epoch = 0
history = []
patience_counter = 0
run_dir = Path(TRAIN_ARGS["project"]) / TRAIN_ARGS["name"]
save_dir = run_dir / "weights"
metrics_dir = run_dir / "metrics"
summary_dir = run_dir / "summary"
save_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)
summary_dir.mkdir(parents=True, exist_ok=True)

best_ckpt_path = save_dir / "best_xai_state_dict.pt"
last_ckpt_path = save_dir / "last_xai_state_dict.pt"
history_json_path = save_dir / "train_history.json"
history_csv_path = save_dir / "train_history.csv"
run_config_path = summary_dir / "run_config.json"
best_val_metrics_path = metrics_dir / "best_val_metrics.json"
best_test_metrics_path = metrics_dir / "best_test_metrics.json"
run_summary_path = summary_dir / "run_summary.json"

save_json(
    {
        "cfg": CFG,
        "train_args": TRAIN_ARGS,
        "val_args": VAL_ARGS,
        "xai_config": {
            "xai_method": xai_cfg.xai_method,
            "lambda_saliency": xai_cfg.lambda_saliency,
            "num_classes": xai_cfg.num_classes,
            "prefer_branch": xai_cfg.prefer_branch,
            "use_progressive_lambda": xai_cfg.use_progressive_lambda,
            "progressive_warmup_epochs": xai_cfg.progressive_warmup_epochs,
            "gt_mask_mode": xai_cfg.gt_mask_mode,
            "soft_mask_sigma": xai_cfg.soft_mask_sigma,
            "shrunk_box_ratio": xai_cfg.shrunk_box_ratio,
        },
        "init_weights": str(INIT_WEIGHTS),
        "runtime_device": str(runtime_device),
    },
    run_config_path,
)

for epoch in range(TRAIN_ARGS["epochs"]):
    epoch_start = time.time()
    step_output = None
    epoch_det_losses = []
    epoch_sal_losses = []
    epoch_total_losses = []
    epoch_eib = []
    epoch_pga = []
    epoch_siou = []
    num_batches = 0

    for batch in train_loader:
        batch = detector_trainer.preprocess_batch(batch)
        step_output = trainer.training_step(batch, epoch=epoch)
        num_batches += 1
        epoch_det_losses.append(float(step_output.detection_loss.item()))
        epoch_sal_losses.append(float(step_output.saliency_loss.item()))
        epoch_total_losses.append(float(step_output.total_loss.item()))
        epoch_eib.append(float(energy_in_box(step_output.saliency_maps, step_output.gt_masks).mean().item()))
        epoch_pga.append(float(pointing_game_accuracy(step_output.saliency_maps, step_output.gt_masks).mean().item()))
        epoch_siou.append(float(saliency_iou(step_output.saliency_maps, step_output.gt_masks).mean().item()))

    if step_output is None:
        raise RuntimeError("train_loader khong tra ve batch nao. Hay kiem tra lai dataset train.")

    train_det = sum(epoch_det_losses) / len(epoch_det_losses)
    train_sal = sum(epoch_sal_losses) / len(epoch_sal_losses)
    train_total = sum(epoch_total_losses) / len(epoch_total_losses)
    train_eib = sum(epoch_eib) / len(epoch_eib)
    train_pga = sum(epoch_pga) / len(epoch_pga)
    train_siou = sum(epoch_siou) / len(epoch_siou)
    current_lr = float(optimizer.param_groups[0]["lr"])
    current_lambda = float(step_output.lambda_saliency)

    current_state = copy.deepcopy(xai_model.state_dict())
    torch.save(current_state, last_ckpt_path)

    eval_model = build_eval_model(current_state)
    val_results = eval_model.val(**VAL_ARGS, split="val")
    val_metrics = extract_eval_metrics(val_results)
    epoch_seconds = time.time() - epoch_start

    epoch_record = {
        "epoch": epoch + 1,
        "num_batches": num_batches,
        "lr": current_lr,
        "lambda_saliency": current_lambda,
        "train_detection_loss": train_det,
        "train_saliency_loss": train_sal,
        "train_total_loss": train_total,
        "train_energy_in_box": train_eib,
        "train_pointing_game": train_pga,
        "train_saliency_iou": train_siou,
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_map50": val_metrics["map50"],
        "val_map50_95": val_metrics["map50_95"],
        "val_fitness": val_metrics["fitness"],
        "val_results_dir": val_metrics["save_dir"],
        "epoch_seconds": epoch_seconds,
    }
    history.append(epoch_record)

    save_json(history, history_json_path)
    with history_csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)

    epoch_metrics_payload = {
        "epoch": epoch + 1,
        "train": {
            "num_batches": num_batches,
            "lr": current_lr,
            "lambda_saliency": current_lambda,
            "detection_loss": train_det,
            "saliency_loss": train_sal,
            "total_loss": train_total,
            "energy_in_box": train_eib,
            "pointing_game": train_pga,
            "saliency_iou": train_siou,
        },
        "val": val_metrics,
        "epoch_seconds": epoch_seconds,
    }
    save_json(epoch_metrics_payload, metrics_dir / f"epoch_{epoch + 1:03d}_val_metrics.json")

    is_best = val_metrics["map50_95"] > best_metric
    if is_best:
        best_metric = val_metrics["map50_95"]
        best_epoch = epoch + 1
        best_state = copy.deepcopy(current_state)
        torch.save(best_state, best_ckpt_path)
        save_json(epoch_metrics_payload, best_val_metrics_path)
        patience_counter = 0
    else:
        patience_counter += 1

    print(f"\n[Epoch {epoch + 1}/{TRAIN_ARGS['epochs']}]")
    print(
        f"train  batches={num_batches} lr={current_lr:.2e} lambda={current_lambda:.4f} "
        f"det={train_det:.4f} sal={train_sal:.4f} total={train_total:.4f}"
    )
    print(
        f"xai    eib={train_eib:.4f} pga={train_pga:.4f} siou={train_siou:.4f}"
    )
    print(
        f"val    precision={val_metrics['precision']:.4f} recall={val_metrics['recall']:.4f} "
        f"mAP50={val_metrics['map50']:.4f} mAP50-95={val_metrics['map50_95']:.4f} "
        f"fitness={val_metrics['fitness']:.4f}"
    )
    print(
        f"state  best_mAP50-95={best_metric:.4f} best_epoch={best_epoch} "
        f"patience={patience_counter}/{TRAIN_ARGS['patience']} time={epoch_seconds:.1f}s"
    )
    print(
        f"files  last_ckpt={last_ckpt_path.name} best_ckpt={best_ckpt_path.name} val_dir={val_metrics['save_dir']}"
    )

    if patience_counter >= TRAIN_ARGS["patience"]:
        print(f"Early stopping at epoch {epoch + 1}")
        break

if best_state is not None:
    xai_model.load_state_dict(best_state)
    best_eval_model = build_eval_model(best_state)
    best_val_results = best_eval_model.val(**VAL_ARGS, split="val")
    best_test_results = best_eval_model.val(**VAL_ARGS, split="test")
    best_val_metrics = extract_eval_metrics(best_val_results)
    best_test_metrics = extract_eval_metrics(best_test_results)
    save_json(best_val_metrics, best_val_metrics_path)
    save_json(best_test_metrics, best_test_metrics_path)
    save_json(
        {
            "best_epoch": best_epoch,
            "best_val_map50_95": best_metric,
            "num_epochs_completed": len(history),
            "best_checkpoint": str(best_ckpt_path),
            "last_checkpoint": str(last_ckpt_path),
            "history_csv": str(history_csv_path),
            "history_json": str(history_json_path),
            "best_val_metrics": best_val_metrics,
            "best_test_metrics": best_test_metrics,
        },
        run_summary_path,
    )

    print("\n[Final Evaluation]")
    print(
        f"best val   precision={best_val_metrics['precision']:.4f} recall={best_val_metrics['recall']:.4f} "
        f"mAP50={best_val_metrics['map50']:.4f} mAP50-95={best_val_metrics['map50_95']:.4f}"
    )
    print(
        f"best test  precision={best_test_metrics['precision']:.4f} recall={best_test_metrics['recall']:.4f} "
        f"mAP50={best_test_metrics['map50']:.4f} mAP50-95={best_test_metrics['map50_95']:.4f}"
    )
    print(f"artifacts  summary={run_summary_path} test_metrics={best_test_metrics_path}")

trainer.close()
